In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.coordinates as coord
import astropy.units as u
import pandas as pd
import sys
from scipy.special import factorial
from astropy.table import Table
from numpy.polynomial.polynomial import Polynomial
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib
from sklearn.decomposition import FastICA

if './SelfCalGroupFinder/py/' not in sys.path:
    sys.path.append('./SelfCalGroupFinder/py/')
from pyutils import *
import plotting as pp
from dataloc import *
from bgs_helpers import *
import catalog_definitions as cat
from groupcatalog import *
from halosimutils import *

%load_ext autoreload
%autoreload 2

In [ ]:
BOX_SIZE_MPC_H = 400  # Mpc/h, from the smdpl documentation
BOX_VOLUME = BOX_SIZE_MPC_H**3  # (Mpc/h)^3

feature_cols = ['LOGMHALO', 'c', 'Spin', 'Halfmass_Scale']
n_features = len(feature_cols) # No dimensionality reduction right now. 

# Define file paths
directory = '/mount/sirocco1/imw2293/GROUP_CAT/DATA/POPMOCK/'
file = directory + 'smdpl_z0.19717.M1E9.C.h5'

In [ ]:
# Read the entire HDF5 file into memory
df = pd.read_hdf(file, key='halos')
df = df[df['upid'] == -1] # Make sure it's only centrals

In [ ]:
# I think rvir is OK for this c calculation
df['c'] = df['rvir'] / df['rs'] # both in kpc/h, so this is dimensionless
df['LOGMHALO'] = np.log10(df['M200b']) # Mpeak instead of M200b? Sometimes it way higher for some reason. Keep M200b

In [ ]:
df_down = downsample_halos(df)

In [ ]:
# Seperate into test/training sets
from sklearn.model_selection import train_test_split
train, test = train_test_split(df_down, test_size=0.1, random_state=894)

# If we add more properties and want to reduce dimensionality, we can change this, but we should see if the recovery tests below still pass and decide what is OK for us.

# Drop rows with any NaN in the selected features
train_clean = train.dropna(subset=feature_cols)
test_clean = test.dropna(subset=feature_cols)
all_clean = df.dropna(subset=feature_cols)
print(f"Training samples after NaN drop: {len(train_clean)} / {len(train)}")
print(f"Test samples after NaN drop: {len(test_clean)} / {len(test)}")
print(f"All samples after NaN drop: {len(all_clean)} / {len(df)}")

In [ ]:
# Scale first - critical since features have very different units/ranges
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_clean[feature_cols])
test_scaled = scaler.transform(test_clean[feature_cols])
all_scaled = scaler.transform(all_clean[feature_cols])

In [ ]:
# Fit full PCA on training set
ica = FastICA(n_components=n_features, whiten='unit-variance', random_state=42, max_iter=5000)
train_ica = ica.fit_transform(train_scaled)

# Reorder: sort ICs by abs correlation with LOGMHALO (descending)
# Fix sign: IC most correlated with LOGMHALO should have positive correlation
corr = np.corrcoef(train_ica.T, train_clean['LOGMHALO'].values)[:-1, -1]
order = np.argsort(-np.abs(corr))
signs = np.sign(corr[order])
# Apply permutation + sign flip to ica.components_ and ica.mixing_
ica.components_ = (ica.components_[order].T * signs).T
ica.mixing_ = (ica.mixing_.T[order].T) * signs  # update mixing accordingly
# TODO I'm confused if I need to use order and signs elsewhere or if this took care of it

del train_ica # Done with this now that we've fixed the order and signs and have the model.

In [ ]:
print(order, signs)

In [ ]:
# Print feature loadings for top components
print("\nFeature loadings:")
loadings = pd.DataFrame(
    ica.components_.T,
    index=feature_cols,
    columns=[f'IC{i+1}' for i in range(ica.components_.shape[0])]
)
print(loadings.round(3).to_string())

IC1 is high halo mass, almost entirly
IC2 is low concentration, and early forming
IC3 is early forming and high spin
IC4 is high spin, almost entirely

In [ ]:
from sklearn.feature_selection import mutual_info_regression
from sklearn.decomposition import PCA
from scipy.stats import ks_2samp

# Fit PCA on same training data for comparison
pca = PCA(n_components=n_features)
pca.fit(train_scaled)
test_pca = pca.transform(test_scaled)

In [ ]:
# Print feature loadings for top components
print("\nFeature loadings:")
loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_cols,
    columns=[f'PC{i+1}' for i in range(pca.components_.shape[0])]
)
print(loadings.round(3).to_string())

In [ ]:
# Apply ICA transformation and join the columns
train_ica_full = ica.transform(train_scaled)
test_ica_full = ica.transform(test_scaled)
all_ica_full = ica.transform(all_scaled)
train_with_ica = pd.concat([train_clean.reset_index(drop=True), pd.DataFrame(train_ica_full, columns=[f'ICA{i+1}' for i in range(train_ica_full.shape[1])])], axis=1)
test_with_ica = pd.concat([test_clean.reset_index(drop=True), pd.DataFrame(test_ica_full, columns=[f'ICA{i+1}' for i in range(test_ica_full.shape[1])])], axis=1)
all_with_ica = pd.concat([all_clean.reset_index(drop=True), pd.DataFrame(all_ica_full, columns=[f'ICA{i+1}' for i in range(all_ica_full.shape[1])])], axis=1)

### ICA vs PCA

In [ ]:
# First let's see if correlation matrix is now diagonal for both
corr_ica = np.corrcoef(test_ica_full, rowvar=False)
corr_pca = np.corrcoef(test_pca, rowvar=False)

# Show
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(corr_ica, vmin=-1, vmax=1, cmap='coolwarm')
plt.colorbar()
plt.title('Correlation Matrix of ICA Components')
plt.subplot(1, 2, 2)
plt.imshow(corr_pca, vmin=-1, vmax=1, cmap='coolwarm')
plt.colorbar()
plt.title('Correlation Matrix of PCA Components')
plt.tight_layout()
plt.show()


In [ ]:
# Pairwise MI for each method
def pairwise_mi(X, random_state=31579):
    n = X.shape[1]
    mi = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                mi[i, j] = mutual_info_regression(
                    X[:, j:j+1], X[:, i], random_state=random_state
                )[0]
    return mi

mi_ica = pairwise_mi(test_ica_full)
mi_pca = pairwise_mi(test_pca)

vmax = max(mi_pca.max(), mi_ica.max())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
labels_ica = [f'IC{i+1}' for i in range(n_features)]
labels_pca = [f'PC{i+1}' for i in range(n_features)]
for ax, mi, labels, title in zip(axes, [mi_pca, mi_ica], [labels_pca, labels_ica], ['PCA', 'ICA']):
    im = ax.imshow(mi, cmap='Oranges', vmin=0, vmax=vmax)
    ax.set_xticks(range(n_features)); ax.set_xticklabels(labels)
    ax.set_yticks(range(n_features)); ax.set_yticklabels(labels)
    for r in range(n_features):
        for c in range(n_features):
            ax.text(c, r, f'{mi[r,c]:.3f}', ha='center', va='center', fontsize=8)
    ax.set_title(f'{title} - Mutual Information')

fig.colorbar(im, ax=axes[1], location='right')
plt.tight_layout()

print(f"ICA mean off-diagonal MI: {mi_ica[mi_ica > 0].mean():.4f}")
print(f"PCA mean off-diagonal MI: {mi_pca[mi_pca > 0].mean():.4f}")

The residual IC2–IC3 dependence you saw (KS=0.110) is physically expected: concentration and formation epoch are genuinely coupled (older halos are more concentrated — this is the physical origin of the mass-concentration relation). This is a real nonlinear correlation in halo physics that no linear transform can fully remove. It corresponds to what people call assembly bias.

In [ ]:
print("Conditional KS tests: P(IC_i | IC_j < median) vs P(IC_i | IC_j >= median)")
print("Large p-value = components behave independently\n")
for method_name, X, labels in [('PCA', test_pca, labels_pca), ('ICA', test_ica_full, labels_ica)]:
    print(f"--- {method_name} ---")
    for j in range(n_features):
        med = np.median(X[:, j])
        for i in range(n_features):
            if i == j: continue
            low  = X[X[:, j] <  med, i]
            high = X[X[:, j] >= med, i]
            stat, p = ks_2samp(low, high)
            flag = "  <-- dependent!" if p < 0.01 else ""
            print(f"  {labels[i]} | {labels[j]} split: KS={stat:.3f}  p={p:.4f}{flag}")
    print()

In [ ]:
# Print number of halos
print(f"Total halos: {len(all_with_ica):,}")

### Save Model


In [ ]:
# Save off the model and scaler for later use
joblib.dump((ica, scaler, feature_cols), HALO_ICA_MODEL_FILE)

# Export ICA model to plain text for C++ consumption
# Format: header lines with # comments, then data blocks
n_comp, n_feat = ica.components_.shape

with open(HALO_ICA_MODEL_TEXT_FILE, 'w') as f:
    f.write(f"# Halo ICA model for C++ use\n")
    f.write(f"# features: {' '.join(feature_cols)}\n")
    f.write(f"# n_features: {n_feat}\n")
    f.write(f"# n_components: {n_comp}\n")
    f.write(f"\n# scaler_mean ({n_feat})\n")
    np.savetxt(f, scaler.mean_.reshape(1, -1), fmt='%.10e')
    f.write(f"\n# scaler_scale ({n_feat})\n")
    np.savetxt(f, scaler.scale_.reshape(1, -1), fmt='%.10e')
    f.write(f"\n# ica_mean_in_scaled_space ({n_feat})\n")
    np.savetxt(f, ica.mean_.reshape(1, -1), fmt='%.10e')
    f.write(f"\n# ica_components ({n_comp} x {n_feat})  — row i is eigenvector i\n")
    np.savetxt(f, ica.components_, fmt='%.10e')
    f.write(f"\n# ica_mixing_matrix ({n_feat} x {n_comp})  — for inverse transform\n")
    np.savetxt(f, ica.mixing_, fmt='%.10e')  # shape (n_feat, n_comp)

print(f"Wrote C++ ICA model to {HALO_ICA_MODEL_TEXT_FILE}")

In [ ]:
def make_adaptive_density_bins(data, n_bins=700, n_tail=30, alpha=0.5):
    """
    Widths go as 1/f(x)^alpha (alpha=0.5: sqrt-density spacing).
    Implemented by spacing edges evenly in cumulative(f^alpha) space.
    Tail extensions use the edge bin spacing for a smooth cutoff to zero.
    """
    N = len(data)
    
    # Pilot density over the covered range (exclude extreme outliers)
    limit = 3e-8  # quantile to exclude from each tail for pilot density
    p_lo, p_hi = np.quantile(data, [limit, 1 - limit])
    n_pilot = min(2000, N // 50)
    pilot_edges = np.linspace(p_lo, p_hi, n_pilot + 1)
    pilot_counts, _ = np.histogram(data, bins=pilot_edges)
    pilot_density = pilot_counts / np.diff(pilot_edges) / N
    # Floor: avoid pathological zero regions swallowing all the tail budget
    floor = pilot_density[pilot_density > 0].min() * 0.01
    pilot_density = np.maximum(pilot_density, floor)
    
    # Cumulative of f^alpha → spacing evenly in this = widths ∝ 1/f^alpha
    dx = np.diff(pilot_edges)
    cumulative = np.concatenate([[0], np.cumsum(pilot_density**alpha * dx)])
    total = cumulative[-1]
    
    target = np.linspace(0, total, n_bins + 1)
    adaptive_edges = np.unique(np.interp(target, cumulative, pilot_edges))
    
    # Extend tails with uniform bins at the edge. Capture some of the rarest halos. Mostly 0's out here.
    dl = np.median(np.diff(adaptive_edges[:10]))
    dr = np.median(np.diff(adaptive_edges[-10:]))
    left  = adaptive_edges[0]  - np.arange(n_tail, 0, -1) * dl
    right = adaptive_edges[-1] + np.arange(1, n_tail + 1)  * dr
    
    return np.concatenate([left, adaptive_edges, right])



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax = axes[0]
ax2 = axes[1]
ax.set_xlabel('ICAx Value')
ax2.set_xlabel('ICAx Value (zoom)')
ax.set_yscale('log')
ax2.set_yscale('log')
ax.set_ylabel('Density [1/(Mpc/h)^3]')
ax2.set_ylabel('Density [1/(Mpc/h)^3]')
ax.grid(True, ls="--", lw=0.5, alpha=0.5)
ax2.grid(True, ls="--", lw=0.5, alpha=0.5)

density_files = [HALO_PCA1_DENSITY_FUNC_FILE, HALO_PCA2_DENSITY_FUNC_FILE,
                 HALO_PCA3_DENSITY_FUNC_FILE, HALO_PCA4_DENSITY_FUNC_FILE]

for i in range(all_ica_full.shape[1]):
    colname = f'ICA{i+1}'
    data = all_with_ica[colname].values

    b = make_adaptive_density_bins(data, alpha=0.35)
    print(f"{colname}: data [{data.min():.2f}, {data.max():.2f}]  "
          f"bins [{b[0]:.2f}, {b[-1]:.2f}]  n_bins={len(b)-1}")

    dICA = np.diff(b)
    counts, _ = np.histogram(data, bins=b)
    centers = 0.5 * (b[:-1] + b[1:])
    density = counts / dICA / BOX_VOLUME
    output_data = np.column_stack((centers, density))

    ax.plot(centers, density, drawstyle='steps-mid', label=colname, color=pp.get_color(i))
    axes[1].plot(centers, density, drawstyle='steps-mid', label=colname, color=pp.get_color(i))
    #np.savetxt(density_files[i], output_data)

ax.legend()
axes[1].set_xlim(-4.5, 3.5)
axes[1].set_ylim(1e-3, 0.9)
plt.tight_layout()
plt.show()

### Tests

In [ ]:
testpoints = 10000
sample = test_clean.iloc[:testpoints]
there = halo_original_to_latent(sample[feature_cols].to_numpy())
latent_cols = [f'ICA{i+1}' for i in range(n_features)]
back = halo_latent_to_original(there)
tol = 1e-5 # more than really needed
assert np.allclose(sample[feature_cols], back, atol=tol), "Roundtrip ICA transformation did not recover original values within tolerance!"

In [ ]:
# Used in a C++ test to verify the model is being loaded and applied correctly there too.
print(all_with_ica.iloc[0])